In [7]:
# =========================================================
# FRAMEWORK MANUAL — BASE ESTRUTURAL
# IMDb + Hubara como primeiro caso
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
import pandas as pd


# ---------------------------------------------------------
# 1) Especificação do cenário
# ---------------------------------------------------------
# Representa o "mundo" que será testado:
# - nome do dataset
# - entidades
# - relacionamentos
# - queries do workload
#
# Exemplo:
# dataset = IMDb
# entities = User, Movie, Rating, ...
@dataclass
class ScenarioSpec:
    name: str
    description: str
    entities: List[str]
    relationships: List[Dict[str, Any]]
    workload_queries: List[Dict[str, Any]]


# ---------------------------------------------------------
# 2) Fragmento recomendado
# ---------------------------------------------------------
# Cada fragmento é um grupo de entidades que será alocado
# em um tipo/modelo de banco.
#
# Exemplo:
# Fragmento 1 = ['User'] -> relational
# Fragmento 2 = ['Movie', 'Genre', ...] -> column
@dataclass
class FragmentSpec:
    name: str
    entities: List[str]
    model: str   # relational, document, column, graph, etc.
    notes: str = ""


# ---------------------------------------------------------
# 3) Recomendação final
# ---------------------------------------------------------
# Representa a saída final de um método como o Hubara.
#
# Exemplo:
# recommendation_name = "Hubara IMDb"
# fragments = [FragmentSpec(...), FragmentSpec(...)]
@dataclass
class RecommendationSpec:
    recommendation_name: str
    source_method: str
    source_dataset: str
    fragments: List[FragmentSpec]


# ---------------------------------------------------------
# 4) Banco físico concreto
# ---------------------------------------------------------
# Aqui diferenciamos "modelo" de "produto".
#
# Exemplo:
# model = relational
# engine = PostgreSQL
#
# ou:
# model = document
# engine = MongoDB
@dataclass
class PhysicalDBSpec:
    model: str
    engine: str
    host: str = "localhost"
    port: Optional[int] = None
    database_name: Optional[str] = None
    username: Optional[str] = None
    password: Optional[str] = None

    # opcionais, úteis para bancos como MongoDB
    connection_uri: Optional[str] = None
    options: Dict[str, Any] = field(default_factory=dict)


# ---------------------------------------------------------
# 5) Plano de materialização
# ---------------------------------------------------------
# Traduz a recomendação abstrata para bancos concretos.
#
# Exemplo:
# fragment User -> PostgreSQL
# fragment Content -> Cassandra
@dataclass
class MaterializationPlan:
    scenario_name: str
    recommendation_name: str
    fragment_to_db: Dict[str, PhysicalDBSpec]


# ---------------------------------------------------------
# 6) Especificação de uma query do benchmark
# ---------------------------------------------------------
# Aqui vamos guardar:
# - nome da query
# - tipo
# - entidades tocadas
# - texto abstrato
#
# Mais tarde, podemos ter versões por banco:
# sql_text, mongo_text, cypher_text, etc.
@dataclass
class BenchmarkQuery:
    name: str
    query_type: str
    entities_involved: List[str]
    abstract_query: str
    expected_fragment: Optional[str] = None
    expected_db_model: Optional[str] = None


# ---------------------------------------------------------
# 7) Especificação do workload
# ---------------------------------------------------------
@dataclass
class WorkloadSpec:
    name: str
    description: str
    queries: List[BenchmarkQuery]
    repetitions: int = 10


# ---------------------------------------------------------
# 8) Resultado de execução de uma query
# ---------------------------------------------------------
@dataclass
class QueryExecutionResult:
    query_name: str
    fragment_name: str
    db_engine: str
    run_id: int
    benchmark_phase: str   # "cold" ou "hot"
    latency_ms: float
    success: bool
    error_message: Optional[str] = None


# ---------------------------------------------------------
# 9) Resultado consolidado do benchmark
# ---------------------------------------------------------
@dataclass
class BenchmarkResult:
    scenario_name: str
    recommendation_name: str
    query_results: List[QueryExecutionResult] = field(default_factory=list)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "scenario_name": self.scenario_name,
                "recommendation_name": self.recommendation_name,
                "query_name": r.query_name,
                "fragment_name": r.fragment_name,
                "db_engine": r.db_engine,
                "run_id": r.run_id,
                "benchmark_phase": r.benchmark_phase,
                "latency_ms": r.latency_ms,
                "success": r.success,
                "error_message": r.error_message,
            }
            for r in self.query_results
        ])

In [8]:
# =========================================================
# INSTÂNCIA 1 — CENÁRIO IMDb + RECOMENDAÇÃO HUBARA
# EXPERIMENTO: ALL POSTGRESQL
# =========================================================

# ---------------------------------------------------------
# 1) Cenário IMDb
# ---------------------------------------------------------
imdb_scenario = ScenarioSpec(
    name="IMDb",
    description="IMDb-like scenario based on the Hubara paper",
    entities=[
        "Person",
        "User",
        "Genre",
        "Role",
        "Rate",
        "WatchItem",
        "Series",
        "Movie",
        "Episode",
    ],
    relationships=[
        {"source": "Person", "target": "Role", "name": "acts_in_via_role"},
        {"source": "Role", "target": "WatchItem", "name": "role_of_watchitem"},
        {"source": "Person", "target": "WatchItem", "name": "directs"},
        {"source": "Person", "target": "WatchItem", "name": "produces"},
        {"source": "Genre", "target": "WatchItem", "name": "has_genre"},
        {"source": "User", "target": "Rate", "name": "rates"},
        {"source": "Rate", "target": "WatchItem", "name": "rate_of_watchitem"},
        {"source": "Series", "target": "WatchItem", "name": "series_is_watchitem"},
        {"source": "Movie", "target": "WatchItem", "name": "movie_is_watchitem"},
        {"source": "Episode", "target": "WatchItem", "name": "episode_is_watchitem"},
        {"source": "Series", "target": "Episode", "name": "series_contains_episode"},
    ],
    workload_queries=[
        {
            "name": "Q1_Login",
            "type": "select",
            "entities": ["User"],
            "abstract_query": "RETURN password FROM User WHERE username = ?"
        },
        {
            "name": "Q2_SimpleSearch",
            "type": "select",
            "entities": ["WatchItem"],
            "abstract_query": "RETURN ALL FROM WatchItem WHERE title = ?"
        },
        {
            "name": "Q3_AddEntitiesAndRelationships",
            "type": "insert",
            "entities": ["Person", "WatchItem", "Role"],
            "abstract_query": "INSERT Person and connect Person with WatchItem"
        },
        {
            "name": "Q4_Recommendation",
            "type": "select",
            "entities": ["WatchItem", "Genre", "Movie", "Series", "Episode"],
            "abstract_query": "Recommendation query by genre and type"
        },
        {
            "name": "Q5_AllPersonsOfTypeForWatchItem",
            "type": "select",
            "entities": ["WatchItem", "Person", "Role"],
            "abstract_query": "RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor"
        },
    ]
)


# ---------------------------------------------------------
# 2) Recomendação Hubara para IMDb
# ---------------------------------------------------------
# Mantemos a fragmentação lógica original:
# - ContentFragment
# - UserFragment
#
# O que muda neste experimento não é a fragmentação lógica,
# mas sim a materialização física: tudo irá para PostgreSQL.
hubara_imdb_recommendation = RecommendationSpec(
    recommendation_name="Hubara_IMDb_Recommendation",
    source_method="Hubara",
    source_dataset="IMDb",
    fragments=[
        FragmentSpec(
            name="ContentFragment",
            entities=["Episode", "Genre", "Movie", "Person", "Rate", "Role", "Series", "WatchItem"],
            model="column",
            notes="Original logical recommendation reproduced from the implemented Hubara workflow"
        ),
        FragmentSpec(
            name="UserFragment",
            entities=["User"],
            model="relational",
            notes="Original logical recommendation reproduced from the implemented Hubara workflow"
        ),
    ]
)


# =========================================================
# BLOCO R1 — MATERIALIZAÇÃO ALTERNATIVA: ALL POSTGRESQL
# =========================================================

hubara_imdb_all_postgres_materialization = MaterializationPlan(
    scenario_name="IMDb",
    recommendation_name="Hubara_IMDb_AllPostgreSQL",
    fragment_to_db={
        "UserFragment": PhysicalDBSpec(
            model="relational",
            engine="PostgreSQL",
            host="127.0.0.1",
            port=55432,
            database_name="imdb_all_postgres_db",
            username="postgres",
            password="postgres"
        ),
        "ContentFragment": PhysicalDBSpec(
            model="relational",
            engine="PostgreSQL",
            host="127.0.0.1",
            port=55432,
            database_name="imdb_all_postgres_db",
            username="postgres",
            password="postgres"
        ),
    }
)

print("Materialização All PostgreSQL criada com sucesso.")
print(hubara_imdb_all_postgres_materialization)


# ---------------------------------------------------------
# 4) Workload do benchmark — alternativa All PostgreSQL
# ---------------------------------------------------------
imdb_workload_all_postgres = WorkloadSpec(
    name="IMDb_Hubara_Workload_AllPostgreSQL",
    description="Benchmark workload for the IMDb scenario using a fully relational PostgreSQL deployment",
    queries=[
        BenchmarkQuery(
            name="Q1_Login",
            query_type="select",
            entities_involved=["User"],
            abstract_query="RETURN password FROM User WHERE username = ?",
            expected_fragment="UserFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q2_SimpleSearch",
            query_type="select",
            entities_involved=["WatchItem"],
            abstract_query="RETURN ALL FROM WatchItem WHERE title = ?",
            expected_fragment="ContentFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q3_AddEntitiesAndRelationships",
            query_type="insert",
            entities_involved=["Person", "WatchItem", "Role"],
            abstract_query="INSERT Person and connect Person with WatchItem",
            expected_fragment="ContentFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q4_Recommendation",
            query_type="select",
            entities_involved=["WatchItem", "Genre", "Movie", "Series", "Episode"],
            abstract_query="Recommendation query by genre and type",
            expected_fragment="ContentFragment",
            expected_db_model="relational"
        ),
        BenchmarkQuery(
            name="Q5_AllPersonsOfTypeForWatchItem",
            query_type="select",
            entities_involved=["WatchItem", "Person", "Role"],
            abstract_query="RETURN Person.ALL FROM WatchItem, Person WHERE title = ? AND rel = Actor",
            expected_fragment="ContentFragment",
            expected_db_model="relational"
        ),
    ],
    repetitions=10
)

Materialização All PostgreSQL criada com sucesso.
MaterializationPlan(scenario_name='IMDb', recommendation_name='Hubara_IMDb_AllPostgreSQL', fragment_to_db={'UserFragment': PhysicalDBSpec(model='relational', engine='PostgreSQL', host='127.0.0.1', port=55432, database_name='imdb_all_postgres_db', username='postgres', password='postgres', connection_uri=None, options={}), 'ContentFragment': PhysicalDBSpec(model='relational', engine='PostgreSQL', host='127.0.0.1', port=55432, database_name='imdb_all_postgres_db', username='postgres', password='postgres', connection_uri=None, options={})})


In [9]:
# =========================================================
# VISUALIZAÇÃO DAS ESTRUTURAS — ALTERNATIVA ALL POSTGRESQL
# =========================================================

# ---------------------------------------------------------
# 1) Entidades do cenário
# ---------------------------------------------------------
scenario_entities_df = pd.DataFrame({"entity": imdb_scenario.entities})

# ---------------------------------------------------------
# 2) Relacionamentos do cenário
# ---------------------------------------------------------
scenario_relationships_df = pd.DataFrame(imdb_scenario.relationships)

# ---------------------------------------------------------
# 3) Recommendation view
# Aqui usamos:
# - as entidades dos fragmentos vindas da recomendação lógica
# - o modelo vindo da materialização alternativa All PostgreSQL
# ---------------------------------------------------------
fragment_entities_map = {
    frag.name: frag.entities
    for frag in hubara_imdb_recommendation.fragments
}

recommendation_df = pd.DataFrame([
    {
        "fragment": fragment_name,
        "entities": fragment_entities_map.get(fragment_name, []),
        "model": dbspec.model,
        "engine": dbspec.engine,
        "notes": f"Alternative physical configuration using {dbspec.engine}"
    }
    for fragment_name, dbspec in hubara_imdb_all_postgres_materialization.fragment_to_db.items()
])

# ---------------------------------------------------------
# 4) Materialization plan
# ---------------------------------------------------------
materialization_df = pd.DataFrame([
    {
        "fragment": fragment_name,
        "model": dbspec.model,
        "engine": dbspec.engine,
        "host": dbspec.host,
        "port": dbspec.port,
        "database_name": dbspec.database_name
    }
    for fragment_name, dbspec in hubara_imdb_all_postgres_materialization.fragment_to_db.items()
])

# ---------------------------------------------------------
# 5) Workload view
# Aqui usamos o workload específico da alternativa All PostgreSQL
# ---------------------------------------------------------
workload_df = pd.DataFrame([
    {
        "query_name": q.name,
        "query_type": q.query_type,
        "entities_involved": q.entities_involved,
        "expected_fragment": q.expected_fragment,
        "expected_db_model": q.expected_db_model,
        "abstract_query": q.abstract_query
    }
    for q in imdb_workload_all_postgres.queries
])

# ---------------------------------------------------------
# 6) Exibição
# ---------------------------------------------------------
print("Entities")
display(scenario_entities_df)

print("Relationships")
display(scenario_relationships_df)

print("Recommendation")
display(recommendation_df)

print("Materialization plan")
display(materialization_df)

print("Workload")
display(workload_df)

Entities


,entity
0,Person
1,User
2,Genre
3,Role
4,Rate
5,WatchItem
6,Series
7,Movie
8,Episode


Relationships


,source,target,name
0,Person,Role,acts_in_via_role
1,Role,WatchItem,role_of_watchitem
2,Person,WatchItem,directs
3,Person,WatchItem,produces
4,Genre,WatchItem,has_genre
5,User,Rate,rates
6,Rate,WatchItem,rate_of_watchitem
7,Series,WatchItem,series_is_watchitem
8,Movie,WatchItem,movie_is_watchitem
9,Episode,WatchItem,episode_is_watchitem


Recommendation


,fragment,entities,model,engine,notes
0,UserFragment,[User],relational,PostgreSQL,Alternative physical configuration using Postg...
1,ContentFragment,"[Episode, Genre, Movie, Person, Rate, Role, Se...",relational,PostgreSQL,Alternative physical configuration using Postg...


Materialization plan


,fragment,model,engine,host,port,database_name
0,UserFragment,relational,PostgreSQL,127.0.0.1,55432,imdb_all_postgres_db
1,ContentFragment,relational,PostgreSQL,127.0.0.1,55432,imdb_all_postgres_db


Workload


,query_name,query_type,entities_involved,expected_fragment,expected_db_model,abstract_query
0,Q1_Login,select,[User],UserFragment,relational,RETURN password FROM User WHERE username = ?
1,Q2_SimpleSearch,select,[WatchItem],ContentFragment,relational,RETURN ALL FROM WatchItem WHERE title = ?
2,Q3_AddEntitiesAndRelationships,insert,"[Person, WatchItem, Role]",ContentFragment,relational,INSERT Person and connect Person with WatchItem
3,Q4_Recommendation,select,"[WatchItem, Genre, Movie, Series, Episode]",ContentFragment,relational,Recommendation query by genre and type
4,Q5_AllPersonsOfTypeForWatchItem,select,"[WatchItem, Person, Role]",ContentFragment,relational,"RETURN Person.ALL FROM WatchItem, Person WHERE..."


In [10]:
# =========================================================
# BLOCO 1 — GERADOR DE DADOS SINTÉTICOS IMDb
# =========================================================

from dataclasses import dataclass, field
from typing import Dict, List
import pandas as pd

@dataclass
class ScenarioDataBundle:
    dataset_name: str
    tables: Dict[str, pd.DataFrame] = field(default_factory=dict)

    def table_names(self) -> List[str]:
        return list(self.tables.keys())

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "table_name": name,
                "rows": len(df),
                "columns": list(df.columns)
            }
            for name, df in self.tables.items()
        ]).sort_values("table_name").reset_index(drop=True)

In [11]:
# =========================================================
# BLOCO 2 — GERADOR DE DADOS SINTÉTICOS IMDb (CORRIGIDO)
# =========================================================

import random
import numpy as np
import pandas as pd

def generate_imdb_synthetic_data(
    n_users: int = 100,
    n_persons: int = 120,
    n_watchitems: int = 80,
    n_genres: int = 10,
    seed: int = 42
) -> "ScenarioDataBundle":
    """
    Gera um dataset sintético simplificado para o cenário IMDb.
    """

    rng = random.Random(seed)
    np_rng = np.random.default_rng(seed)

    # 1) USERS
    users = pd.DataFrame([
        {
            "user_id": i,
            "username": f"user_{i}",
            "password": f"pass_{i}",
            "email": f"user_{i}@mail.com",
            "last_login": f"2026-01-{(i % 28) + 1:02d}"
        }
        for i in range(1, n_users + 1)
    ])

    # 2) PERSONS
    persons = pd.DataFrame([
        {
            "person_id": i,
            "name": f"Person {i}",
            "date_of_birth": f"{1970 + (i % 30)}-{(i % 12) + 1:02d}-{(i % 28) + 1:02d}",
            "gender": "M" if i % 2 == 0 else "F"
        }
        for i in range(1, n_persons + 1)
    ])

    # 3) GENRES
    genre_names = [
        "Action", "Drama", "Comedy", "Thriller", "SciFi",
        "Fantasy", "Romance", "Crime", "Adventure", "Mystery"
    ][:n_genres]

    genres = pd.DataFrame([
        {"genre_id": i + 1, "name": genre_names[i]}
        for i in range(len(genre_names))
    ])

    # 4) WATCHITEMS
    item_types = np_rng.choice(
        ["Movie", "Series", "Episode"],
        size=n_watchitems,
        p=[0.45, 0.20, 0.35]
    )

    watchitems_rows = []
    for i in range(1, n_watchitems + 1):
        genre_id = int(np_rng.integers(1, len(genres) + 1))
        release_year = int(np_rng.integers(1990, 2026))

        watchitems_rows.append({
            "watchitem_id": i,
            "title": f"Title {i}",
            "release_year": release_year,
            "avg_rating": round(float(np_rng.uniform(1.0, 10.0)), 2),
            "genre_id": genre_id,
            "item_type": item_types[i - 1]
        })

    watchitems = pd.DataFrame(watchitems_rows)

    # 5) MOVIES
    movies = watchitems[watchitems["item_type"] == "Movie"][["watchitem_id"]].copy()
    movies["length_min"] = np_rng.integers(80, 181, size=len(movies))
    movies["media"] = np_rng.choice(["Cinema", "Streaming", "TV"], size=len(movies))
    movies["income"] = np_rng.integers(100000, 1000000000, size=len(movies))

    # 6) SERIES
    series = watchitems[watchitems["item_type"] == "Series"][["watchitem_id"]].copy()
    series["seasons"] = np_rng.integers(1, 8, size=len(series))
    series["network"] = np_rng.choice(["HBO", "Netflix", "Prime", "Disney"], size=len(series))

    # 7) EPISODES
    episodes = watchitems[watchitems["item_type"] == "Episode"][["watchitem_id"]].copy()
    episodes["season"] = np_rng.integers(1, 8, size=len(episodes))
    episodes["episode_number"] = np_rng.integers(1, 25, size=len(episodes))
    episodes["length_min"] = np_rng.integers(20, 61, size=len(episodes))

    if len(series) > 0 and len(episodes) > 0:
        series_ids = series["watchitem_id"].tolist()
        episodes["series_watchitem_id"] = [rng.choice(series_ids) for _ in range(len(episodes))]
    else:
        episodes["series_watchitem_id"] = None

    # 8) ROLES
    role_types = ["Actor", "Director", "Producer"]

    roles_rows = []
    role_id = 1
    for watchitem_id in watchitems["watchitem_id"]:
        n_links = int(np_rng.integers(2, 6))
        chosen_persons = rng.sample(persons["person_id"].tolist(), k=min(n_links, len(persons)))

        for person_id in chosen_persons:
            roles_rows.append({
                "role_id": role_id,
                "person_id": person_id,
                "watchitem_id": watchitem_id,
                "role_type": rng.choice(role_types)
            })
            role_id += 1

    roles = pd.DataFrame(roles_rows)

    # 9) RATES
    rates_rows = []
    rate_id = 1

    for user_id in users["user_id"]:
        n_user_ratings = int(np_rng.integers(2, 10))
        chosen_items = rng.sample(
            watchitems["watchitem_id"].tolist(),
            k=min(n_user_ratings, len(watchitems))
        )

        for watchitem_id in chosen_items:
            rating_value = int(np_rng.integers(1, 11))
            verbal = (
                "Excellent" if rating_value >= 9 else
                "Good" if rating_value >= 7 else
                "Average" if rating_value >= 5 else
                "Bad"
            )

            rates_rows.append({
                "rate_id": rate_id,
                "user_id": user_id,
                "watchitem_id": watchitem_id,
                "rating": rating_value,
                "verbal_rating": verbal
            })
            rate_id += 1

    rates = pd.DataFrame(rates_rows)

    return ScenarioDataBundle(
        dataset_name="IMDb",
        tables={
            "users": users,
            "persons": persons,
            "genres": genres,
            "watchitems": watchitems,
            "movies": movies,
            "series": series,
            "episodes": episodes,
            "roles": roles,
            "rates": rates,
        }
    )

In [12]:
# =========================================================
# BLOCO 3 — GERAR O DATASET IMDb SINTÉTICO
# =========================================================

import pandas as pd

# Checagem simples para evitar erro confuso se o bloco anterior não tiver rodado
if "generate_imdb_synthetic_data" not in globals():
    raise NameError(
        "A função 'generate_imdb_synthetic_data' não está definida. "
        "Rode primeiro o bloco anterior que cria essa função."
    )

# Gera o dataset sintético
imdb_data_bundle = generate_imdb_synthetic_data(
    n_users=100,
    n_persons=120,
    n_watchitems=80,
    n_genres=10,
    seed=42
)

# Mostra um resumo das tabelas geradas
print("Resumo das tabelas geradas:")
display(imdb_data_bundle.summary())

# Mostra algumas amostras para inspeção
print("Prévia: users")
display(imdb_data_bundle.tables["users"].head())

print("Prévia: watchitems")
display(imdb_data_bundle.tables["watchitems"].head())

print("Prévia: roles")
display(imdb_data_bundle.tables["roles"].head())

print("Prévia: rates")
display(imdb_data_bundle.tables["rates"].head())

Resumo das tabelas geradas:


,table_name,rows,columns
0,episodes,31,"[watchitem_id, season, episode_number, length_..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[watchitem_id, length_min, media, income]"
3,persons,120,"[person_id, name, date_of_birth, gender]"
4,rates,547,"[rate_id, user_id, watchitem_id, rating, verba..."
5,roles,290,"[role_id, person_id, watchitem_id, role_type]"
6,series,15,"[watchitem_id, seasons, network]"
7,users,100,"[user_id, username, password, email, last_login]"
8,watchitems,80,"[watchitem_id, title, release_year, avg_rating..."


Prévia: users


,user_id,username,password,email,last_login
0,1,user_1,pass_1,user_1@mail.com,2026-01-02
1,2,user_2,pass_2,user_2@mail.com,2026-01-03
2,3,user_3,pass_3,user_3@mail.com,2026-01-04
3,4,user_4,pass_4,user_4@mail.com,2026-01-05
4,5,user_5,pass_5,user_5@mail.com,2026-01-06


Prévia: watchitems


,watchitem_id,title,release_year,avg_rating,genre_id,item_type
0,1,Title 1,2013,4.66,10,Episode
1,2,Title 2,2019,2.50,5,Movie
2,3,Title 3,1990,1.81,4,Episode
3,4,Title 4,2016,5.16,8,Episode
4,5,Title 5,1995,5.51,8,Movie


Prévia: roles


,role_id,person_id,watchitem_id,role_type
0,1,54,1,Producer
1,2,29,1,Director
2,3,58,1,Actor
3,4,98,2,Producer
4,5,104,2,Director


Prévia: rates


,rate_id,user_id,watchitem_id,rating,verbal_rating
0,1,1,26,1,Bad
1,2,1,47,1,Bad
2,3,1,56,5,Average
3,4,1,9,8,Good
4,5,1,43,9,Excellent


In [13]:
# =========================================================
# BLOCO 4 — PARTICIONAR OS DADOS PELOS FRAGMENTOS
# =========================================================

# Mapa simples: qual tabela lógica pertence a qual entidade conceitual
LOGICAL_TABLE_TO_ENTITY = {
    "users": "User",
    "persons": "Person",
    "genres": "Genre",
    "watchitems": "WatchItem",
    "movies": "Movie",
    "series": "Series",
    "episodes": "Episode",
    "roles": "Role",
    "rates": "Rate",
}


def split_bundle_by_recommendation(
    bundle: ScenarioDataBundle,
    recommendation: RecommendationSpec
) -> Dict[str, Dict[str, pd.DataFrame]]:
    """
    Retorna:
    {
        "UserFragment": {
            "users": df
        },
        "ContentFragment": {
            "persons": df,
            "genres": df,
            ...
        }
    }
    """

    fragment_map = {frag.name: set(frag.entities) for frag in recommendation.fragments}
    result = {frag.name: {} for frag in recommendation.fragments}

    for table_name, df in bundle.tables.items():
        entity_name = LOGICAL_TABLE_TO_ENTITY[table_name]

        for fragment_name, fragment_entities in fragment_map.items():
            if entity_name in fragment_entities:
                result[fragment_name][table_name] = df.copy()
                break

    return result

In [14]:
# =========================================================
# BLOCO 5 — EXECUTAR PARTICIONAMENTO DOS DADOS
# =========================================================

fragmented_data = split_bundle_by_recommendation(
    bundle=imdb_data_bundle,
    recommendation=hubara_imdb_recommendation
)

for fragment_name, tables_dict in fragmented_data.items():
    print("\n" + "=" * 70)
    print(f"Fragmento: {fragment_name}")

    summary_df = pd.DataFrame([
        {
            "table_name": table_name,
            "rows": len(df),
            "columns": list(df.columns)
        }
        for table_name, df in tables_dict.items()
    ]).sort_values("table_name").reset_index(drop=True)

    display(summary_df)


Fragmento: ContentFragment


,table_name,rows,columns
0,episodes,31,"[watchitem_id, season, episode_number, length_..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[watchitem_id, length_min, media, income]"
3,persons,120,"[person_id, name, date_of_birth, gender]"
4,rates,547,"[rate_id, user_id, watchitem_id, rating, verba..."
5,roles,290,"[role_id, person_id, watchitem_id, role_type]"
6,series,15,"[watchitem_id, seasons, network]"
7,watchitems,80,"[watchitem_id, title, release_year, avg_rating..."



Fragmento: UserFragment


,table_name,rows,columns
0,users,100,"[user_id, username, password, email, last_login]"


In [15]:
# =========================================================
# BLOCO 6 — PREPARAÇÃO DE PAYLOAD FÍSICO
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

@dataclass
class PhysicalLoadBundle:
    fragment_name: str
    db_model: str
    db_engine: str
    physical_tables: Dict[str, pd.DataFrame] = field(default_factory=dict)

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame([
            {
                "physical_table": name,
                "rows": len(df),
                "columns": list(df.columns)
            }
            for name, df in self.physical_tables.items()
        ]).sort_values("physical_table").reset_index(drop=True)


def build_user_fragment_postgres_payload(fragment_tables: Dict[str, pd.DataFrame]) -> PhysicalLoadBundle:
    """
    Materialização simples do fragmento User para PostgreSQL.
    """
    users = fragment_tables["users"].copy()

    return PhysicalLoadBundle(
        fragment_name="UserFragment",
        db_model="relational",
        db_engine="PostgreSQL",
        physical_tables={
            "users": users
        }
    )


def build_content_fragment_postgres_payload(fragment_tables: Dict[str, pd.DataFrame]) -> PhysicalLoadBundle:
    """
    Materialização do fragmento de conteúdo para PostgreSQL.

    Estratégia do baseline relacional:
    - manter apenas as tabelas canônicas;
    - deixar as queries do benchmark serem resolvidas por SQL
      usando joins, filtros e índices.
    """

    persons = fragment_tables["persons"].copy()
    genres = fragment_tables["genres"].copy()
    watchitems = fragment_tables["watchitems"].copy()
    movies = fragment_tables["movies"].copy()
    series = fragment_tables["series"].copy()
    episodes = fragment_tables["episodes"].copy()
    roles = fragment_tables["roles"].copy()
    rates = fragment_tables["rates"].copy()

    return PhysicalLoadBundle(
        fragment_name="ContentFragment",
        db_model="relational",
        db_engine="PostgreSQL",
        physical_tables={
            "persons": persons,
            "genres": genres,
            "watchitems": watchitems,
            "movies": movies,
            "series": series,
            "episodes": episodes,
            "roles": roles,
            "rates": rates,
        }
    )

In [16]:
# =========================================================
# BLOCO 7 — CONSTRUIR PAYLOADS FÍSICOS
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

# Fragmento de usuário continua no PostgreSQL
user_payload_all_postgres = build_user_fragment_postgres_payload(
    fragmented_data["UserFragment"]
)

# Fragmento de conteúdo agora também vai para PostgreSQL
content_payload_all_postgres = build_content_fragment_postgres_payload(
    fragmented_data["ContentFragment"]
)

print("Resumo do payload PostgreSQL (UserFragment):")
display(user_payload_all_postgres.summary())

print("Resumo do payload PostgreSQL (ContentFragment):")
display(content_payload_all_postgres.summary())

print("Prévia: users (PostgreSQL)")
display(user_payload_all_postgres.physical_tables["users"].head())

print("Prévia: watchitems (PostgreSQL)")
display(content_payload_all_postgres.physical_tables["watchitems"].head())

print("Prévia: roles (PostgreSQL)")
display(content_payload_all_postgres.physical_tables["roles"].head())

print("Prévia: rates (PostgreSQL)")
display(content_payload_all_postgres.physical_tables["rates"].head())

Resumo do payload PostgreSQL (UserFragment):


,physical_table,rows,columns
0,users,100,"[user_id, username, password, email, last_login]"


Resumo do payload PostgreSQL (ContentFragment):


,physical_table,rows,columns
0,episodes,31,"[watchitem_id, season, episode_number, length_..."
1,genres,10,"[genre_id, name]"
2,movies,34,"[watchitem_id, length_min, media, income]"
3,persons,120,"[person_id, name, date_of_birth, gender]"
4,rates,547,"[rate_id, user_id, watchitem_id, rating, verba..."
5,roles,290,"[role_id, person_id, watchitem_id, role_type]"
6,series,15,"[watchitem_id, seasons, network]"
7,watchitems,80,"[watchitem_id, title, release_year, avg_rating..."


Prévia: users (PostgreSQL)


...

,watchitem_id,title,release_year,avg_rating,genre_id,item_type
0,1,Title 1,2013,4.66,10,Episode
1,2,Title 2,2019,2.50,5,Movie
2,3,Title 3,1990,1.81,4,Episode
3,4,Title 4,2016,5.16,8,Episode
4,5,Title 5,1995,5.51,8,Movie


Prévia: roles (PostgreSQL)


,role_id,person_id,watchitem_id,role_type
0,1,54,1,Producer
1,2,29,1,Director
2,3,58,1,Actor
3,4,98,2,Producer
4,5,104,2,Director


Prévia: rates (PostgreSQL)


,rate_id,user_id,watchitem_id,rating,verbal_rating
0,1,1,26,1,Bad
1,2,1,47,1,Bad
2,3,1,56,5,Average
3,4,1,9,8,Good
4,5,1,43,9,Excellent


In [17]:
# =========================================================
# BLOCO A1.1 — UTILITÁRIOS DE EXPORTAÇÃO
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json


# ---------------------------------------------------------
# 1) Criar diretório se não existir
# ---------------------------------------------------------
def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


# ---------------------------------------------------------
# 2) Escapar strings para SQL
# ---------------------------------------------------------
def escape_sql_string(value: str) -> str:
    """
    Troca ' por '' para evitar quebrar INSERTs SQL.
    """
    return value.replace("'", "''")


# ---------------------------------------------------------
# 3) Converter valor Python/Pandas para literal SQL
# ---------------------------------------------------------
def to_sql_literal(value):
    """
    Exemplos:
    - 10        -> 10
    - 3.14      -> 3.14
    - "abc"     -> 'abc'
    - None/NaN  -> NULL
    """
    if pd.isna(value):
        return "NULL"
    if isinstance(value, (int, np.integer)):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        return str(float(value))
    if isinstance(value, bool):
        return "TRUE" if value else "FALSE"
    return f"'{escape_sql_string(str(value))}'"


# ---------------------------------------------------------
# 4) Inferir tipo PostgreSQL simples a partir da coluna
# ---------------------------------------------------------
def infer_postgres_type(series: pd.Series) -> str:
    """
    Inferência simples e pragmática.
    """
    if pd.api.types.is_integer_dtype(series):
        return "BIGINT"
    if pd.api.types.is_float_dtype(series):
        return "DOUBLE PRECISION"
    if pd.api.types.is_bool_dtype(series):
        return "BOOLEAN"

    return "TEXT"


# ---------------------------------------------------------
# 5) Normalizar DataFrame para JSON genérico
# ---------------------------------------------------------
def dataframe_to_json_records(df: pd.DataFrame) -> list[dict]:
    """
    Converte um DataFrame em uma lista de registros Python puros,
    substituindo NaN por None e convertendo tipos numpy para tipos nativos.
    """

    def normalize_value(v):
        if pd.isna(v):
            return None
        if isinstance(v, np.integer):
            return int(v)
        if isinstance(v, np.floating):
            return float(v)
        if isinstance(v, np.bool_):
            return bool(v)
        return v

    records = df.to_dict(orient="records")

    return [
        {k: normalize_value(v) for k, v in record.items()}
        for record in records
    ]


# ---------------------------------------------------------
# 6) Salvar texto em arquivo
# ---------------------------------------------------------
def write_text_file(path: str | Path, content: str) -> Path:
    path = Path(path)
    path.write_text(content, encoding="utf-8")
    return path

In [18]:
# =========================================================
# BLOCO A1.2 — EXPORTAR TABELAS FÍSICAS PARA CSV E JSON
# =========================================================

def export_bundle_to_csv_json(
    payload: PhysicalLoadBundle,
    base_output_dir: str | Path
) -> dict[str, list[Path]]:
    """
    Exporta cada tabela física do payload para:
    - CSV
    - JSON (records)

    Estrutura:
    base_output_dir/
        fragment_name/
            csv/
            json/
    """

    base_output_dir = ensure_dir(base_output_dir)
    fragment_dir = ensure_dir(base_output_dir / payload.fragment_name)
    csv_dir = ensure_dir(fragment_dir / "csv")
    json_dir = ensure_dir(fragment_dir / "json")

    exported_csv = []
    exported_json = []

    for table_name, df in payload.physical_tables.items():
        csv_path = csv_dir / f"{table_name}.csv"
        json_path = json_dir / f"{table_name}.json"

        # CSV
        df.to_csv(csv_path, index=False)

        # JSON
        json_path.write_text(
            json.dumps(
                dataframe_to_json_records(df),
                indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

        exported_csv.append(csv_path)
        exported_json.append(json_path)

    return {
        "csv": exported_csv,
        "json": exported_json,
    }

In [19]:
# =========================================================
# BLOCO A1.3 — GERAR SCRIPT SQL PARA POSTGRESQL
# =========================================================

def build_postgres_create_table_sql(
    table_name: str,
    df: pd.DataFrame,
    primary_key: str | None = None
) -> str:
    """
    Gera um CREATE TABLE simples para PostgreSQL.
    """
    column_defs = []

    for col in df.columns:
        pg_type = infer_postgres_type(df[col])

        if primary_key is not None and col == primary_key:
            column_defs.append(f'    "{col}" {pg_type} PRIMARY KEY')
        else:
            column_defs.append(f'    "{col}" {pg_type}')

    sql = f'CREATE TABLE IF NOT EXISTS "{table_name}" (\n'
    sql += ",\n".join(column_defs)
    sql += "\n);\n"

    return sql


def build_postgres_insert_sql(table_name: str, df: pd.DataFrame) -> str:
    """
    Gera INSERTs simples para PostgreSQL.
    """
    columns_sql = ", ".join([f'"{col}"' for col in df.columns])

    lines = []
    for _, row in df.iterrows():
        values_sql = ", ".join([to_sql_literal(row[col]) for col in df.columns])
        lines.append(f'INSERT INTO "{table_name}" ({columns_sql}) VALUES ({values_sql});')

    return "\n".join(lines) + "\n"


def export_postgres_payload_sql(
    payload: PhysicalLoadBundle,
    output_dir: str | Path,
    primary_keys: dict[str, str] | None = None
) -> Path:
    """
    Exporta um único arquivo .sql contendo CREATE TABLE + INSERT
    para um único payload.
    """
    if primary_keys is None:
        primary_keys = {}

    output_dir = ensure_dir(output_dir)
    sql_parts = []

    for table_name, df in payload.physical_tables.items():
        pk = primary_keys.get(table_name)

        sql_parts.append(f"-- ==================================================")
        sql_parts.append(f"-- FRAGMENT: {payload.fragment_name}")
        sql_parts.append(f"-- TABLE: {table_name}")
        sql_parts.append(f"-- ==================================================\n")

        sql_parts.append(build_postgres_create_table_sql(table_name, df, primary_key=pk))
        sql_parts.append(build_postgres_insert_sql(table_name, df))
        sql_parts.append("\n")

    final_sql = "\n".join(sql_parts)

    out_path = output_dir / f"{payload.fragment_name}_postgres.sql"
    write_text_file(out_path, final_sql)
    return out_path


def export_multiple_postgres_payloads_sql(
    payloads: list[PhysicalLoadBundle],
    output_dir: str | Path,
    output_filename: str,
    primary_keys_by_table: dict[str, str] | None = None
) -> Path:
    """
    Exporta vários payloads PostgreSQL para um único arquivo .sql.

    Útil para o experimento All PostgreSQL, em que UserFragment e
    ContentFragment serão carregados no mesmo database.
    """
    if primary_keys_by_table is None:
        primary_keys_by_table = {}

    output_dir = ensure_dir(output_dir)
    sql_parts = []

    for payload in payloads:
        for table_name, df in payload.physical_tables.items():
            pk = primary_keys_by_table.get(table_name)

            sql_parts.append(f"-- ==================================================")
            sql_parts.append(f"-- FRAGMENT: {payload.fragment_name}")
            sql_parts.append(f"-- TABLE: {table_name}")
            sql_parts.append(f"-- ==================================================\n")

            sql_parts.append(build_postgres_create_table_sql(table_name, df, primary_key=pk))
            sql_parts.append(build_postgres_insert_sql(table_name, df))
            sql_parts.append("\n")

    final_sql = "\n".join(sql_parts)

    out_path = output_dir / output_filename
    write_text_file(out_path, final_sql)
    return out_path

In [20]:
# =========================================================
# BLOCO A1.5 — EXECUTAR EXPORTAÇÃO COMPLETA
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

# ---------------------------------------------------------
# 1) Checagens simples
# ---------------------------------------------------------
required_names = [
    "user_payload_all_postgres",
    "content_payload_all_postgres",
    "export_bundle_to_csv_json",
    "export_postgres_payload_sql",
    "export_multiple_postgres_payloads_sql",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 2) Diretório-base de saída
# ---------------------------------------------------------
EXPORT_BASE_DIR = ensure_dir("exports/imdb_hubara_all_postgres")

# ---------------------------------------------------------
# 3) Exportar CSV/JSON
# ---------------------------------------------------------
user_files = export_bundle_to_csv_json(user_payload_all_postgres, EXPORT_BASE_DIR)
content_files = export_bundle_to_csv_json(content_payload_all_postgres, EXPORT_BASE_DIR)

print("Arquivos CSV/JSON exportados para UserFragment:")
for kind, paths in user_files.items():
    print(f"\n{kind.upper()}:")
    for p in paths:
        print(p)

print("\nArquivos CSV/JSON exportados para ContentFragment:")
for kind, paths in content_files.items():
    print(f"\n{kind.upper()}:")
    for p in paths:
        print(p)

# ---------------------------------------------------------
# 4) Definir primary keys PostgreSQL
# ---------------------------------------------------------
postgres_primary_keys = {
    "users": "user_id",
    "persons": "person_id",
    "genres": "genre_id",
    "watchitems": "watchitem_id",
    "movies": "watchitem_id",
    "series": "watchitem_id",
    "episodes": "watchitem_id",
    "roles": "role_id",
    "rates": "rate_id",
}

# ---------------------------------------------------------
# 5) Exportar SQL separado por fragmento
# ---------------------------------------------------------
user_postgres_sql_path = export_postgres_payload_sql(
    payload=user_payload_all_postgres,
    output_dir=EXPORT_BASE_DIR / "sql",
    primary_keys=postgres_primary_keys
)

content_postgres_sql_path = export_postgres_payload_sql(
    payload=content_payload_all_postgres,
    output_dir=EXPORT_BASE_DIR / "sql",
    primary_keys=postgres_primary_keys
)

print("\nScript SQL PostgreSQL do UserFragment gerado:")
print(user_postgres_sql_path)

print("\nScript SQL PostgreSQL do ContentFragment gerado:")
print(content_postgres_sql_path)

# ---------------------------------------------------------
# 6) Exportar SQL consolidado (mais útil para carga real)
# ---------------------------------------------------------
all_postgres_sql_path = export_multiple_postgres_payloads_sql(
    payloads=[user_payload_all_postgres, content_payload_all_postgres],
    output_dir=EXPORT_BASE_DIR / "sql",
    output_filename="all_postgres_combined.sql",
    primary_keys_by_table=postgres_primary_keys
)

print("\nScript SQL PostgreSQL consolidado gerado:")
print(all_postgres_sql_path)

Arquivos CSV/JSON exportados para UserFragment:

CSV:
exports/imdb_hubara_all_postgres/UserFragment/csv/users.csv

JSON:
exports/imdb_hubara_all_postgres/UserFragment/json/users.json

Arquivos CSV/JSON exportados para ContentFragment:

CSV:
exports/imdb_hubara_all_postgres/ContentFragment/csv/persons.csv
exports/imdb_hubara_all_postgres/ContentFragment/csv/genres.csv
exports/imdb_hubara_all_postgres/ContentFragment/csv/watchitems.csv
exports/imdb_hubara_all_postgres/ContentFragment/csv/movies.csv
exports/imdb_hubara_all_postgres/ContentFragment/csv/series.csv
exports/imdb_hubara_all_postgres/ContentFragment/csv/episodes.csv
exports/imdb_hubara_all_postgres/ContentFragment/csv/roles.csv
exports/imdb_hubara_all_postgres/ContentFragment/csv/rates.csv

JSON:
exports/imdb_hubara_all_postgres/ContentFragment/json/persons.json
exports/imdb_hubara_all_postgres/ContentFragment/json/genres.json
exports/imdb_hubara_all_postgres/ContentFragment/json/watchitems.json
exports/imdb_hubara_all_postgres

In [26]:
# =========================================================
# BLOCO A1.6 — INSPECIONAR OS ARTEFATOS GERADOS
# ALTERNATIVA: POSTGRESQL + MONGODB
# =========================================================

from pathlib import Path
import json

# ---------------------------------------------------------
# 1) Prévia do script PostgreSQL
# ---------------------------------------------------------
print("Prévia do script PostgreSQL:\n")
print(Path(postgres_sql_path).read_text(encoding="utf-8")[:3000])

print("\n" + "=" * 80 + "\n")

# ---------------------------------------------------------
# 2) Prévia de um JSON do ContentFragment (MongoDB)
# ---------------------------------------------------------
sample_json_path = mongo_json_dir / "watchitems_by_title.json"

print(f"Prévia do JSON MongoDB: {sample_json_path}\n")

json_text = Path(sample_json_path).read_text(encoding="utf-8")
print(json_text[:3000])

Prévia do script PostgreSQL:

-- ==================================================
-- TABLE: users
-- ==================================================

CREATE TABLE IF NOT EXISTS "users" (
    "user_id" BIGINT PRIMARY KEY,
    "username" TEXT,
    "password" TEXT,
    "email" TEXT,
    "last_login" TEXT
);

INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (1, 'user_1', 'pass_1', 'user_1@mail.com', '2026-01-02');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (2, 'user_2', 'pass_2', 'user_2@mail.com', '2026-01-03');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (3, 'user_3', 'pass_3', 'user_3@mail.com', '2026-01-04');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (4, 'user_4', 'pass_4', 'user_4@mail.com', '2026-01-05');
INSERT INTO "users" ("user_id", "username", "password", "email", "last_login") VALUES (5, 'user_5', 'pass_5', 'u

In [21]:
pip install psycopg[binary]

Note: you may need to restart the kernel to use updated packages.


In [22]:
# =========================================================
# BLOCO A2.1 — IMPORTS E FUNÇÕES UTILITÁRIAS
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

from pathlib import Path
import re

# PostgreSQL
try:
    import psycopg
    HAS_PSYCOPG3 = True
except ImportError:
    HAS_PSYCOPG3 = False
    psycopg = None


def safe_identifier(name: str) -> str:
    """
    Garante que nomes como database_name e table_name sejam simples e seguros.
    Aceita apenas letras, números e underscore, e não pode começar com número.
    """
    if not re.match(r"^[A-Za-z_][A-Za-z0-9_]*$", name):
        raise ValueError(f"Identificador inválido: {name}")
    return name


def strip_comment_lines(script_text: str) -> str:
    """
    Remove linhas que começam com '--'.
    Isso ajuda a executar scripts SQL statement por statement.
    """
    cleaned_lines = []
    for line in script_text.splitlines():
        stripped = line.strip()
        if stripped.startswith("--"):
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)


def split_script_statements(script_text: str) -> list[str]:
    """
    Divide um script em statements usando ';' como separador.
    Como nossos scripts são simples, isso é suficiente.
    """
    script_text = strip_comment_lines(script_text)
    parts = script_text.split(";")
    statements = []

    for part in parts:
        stmt = part.strip()
        if stmt:
            statements.append(stmt + ";")

    return statements

In [23]:
# =========================================================
# BLOCO A2.2 — POSTGRESQL: CRIAR DATABASE E EXECUTAR SCRIPT
# =========================================================

def pg_connect(dbname: str, spec: PhysicalDBSpec):
    """
    Abre conexão com PostgreSQL usando psycopg3.
    """
    if not HAS_PSYCOPG3:
        raise ImportError(
            "psycopg não está instalado. Rode: pip install psycopg[binary]"
        )

    conn = psycopg.connect(
        host=spec.host,
        port=spec.port,
        dbname=dbname,
        user=spec.username,
        password=spec.password,
        autocommit=True
    )
    return conn


def ensure_postgres_database_exists(spec: PhysicalDBSpec):
    """
    Cria o database alvo se ele ainda não existir.
    Conecta primeiro no database padrão 'postgres'.
    """
    target_db = safe_identifier(spec.database_name)

    with pg_connect("postgres", spec) as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 1 FROM pg_database WHERE datname = %s;", (target_db,))
            exists = cur.fetchone() is not None

            if not exists:
                cur.execute(f'CREATE DATABASE "{target_db}";')
                print(f"Database PostgreSQL criado: {target_db}")
            else:
                print(f"Database PostgreSQL já existe: {target_db}")


def execute_postgres_sql_script(spec: PhysicalDBSpec, sql_script_path: str | Path):
    """
    Executa um arquivo .sql statement por statement no database alvo.
    """
    sql_script_path = Path(sql_script_path)
    target_db = safe_identifier(spec.database_name)

    script_text = sql_script_path.read_text(encoding="utf-8")
    statements = split_script_statements(script_text)

    with pg_connect(target_db, spec) as conn:
        with conn.cursor() as cur:
            for stmt in statements:
                cur.execute(stmt)

    print(f"Script SQL executado com sucesso em PostgreSQL: {sql_script_path}")


def verify_postgres_table_counts(spec: PhysicalDBSpec, table_names: list[str]) -> pd.DataFrame:
    """
    Verifica quantidade de linhas por tabela no PostgreSQL.
    """
    rows = []
    target_db = safe_identifier(spec.database_name)

    with pg_connect(target_db, spec) as conn:
        with conn.cursor() as cur:
            for table_name in table_names:
                safe_table = safe_identifier(table_name)
                cur.execute(f'SELECT COUNT(*) FROM "{safe_table}";')
                count = cur.fetchone()[0]
                rows.append({
                    "table_name": table_name,
                    "row_count": count
                })

    return pd.DataFrame(rows)

In [42]:
# =========================================================
# BLOCO A2.3 — NÃO UTILIZADO NO EXPERIMENTO ALL POSTGRESQL
# =========================================================

# Neste experimento, todo o ambiente é materializado em PostgreSQL.
# Portanto, não há necessidade de um bloco específico para MongoDB
# ou outro segundo SGBD.
#
# A carga real será feita diretamente no PostgreSQL usando:
# - ensure_postgres_database_exists(...)
# - execute_postgres_sql_script(...)
# - verify_postgres_table_counts(...)

In [46]:
#para conectar no servidor remoto - rodar no terminal 
#ssh -N \
#  -L 55432:127.0.0.1:5432 \
#  -L 59042:127.0.0.1:9042 \
#  -L 57017:127.0.0.1:27017 \
#  hudson@150.162.57.138

PostgreSQL: falha na conexão
ConnectionTimeout connection timeout expired
Cassandra: falha na conexão
NoHostAvailable ('Unable to connect to any servers', {'150.162.57.138:9042': OSError(None, "Tried connecting to [('150.162.57.138', 9042)]. Last error: timed out")})


In [24]:
# =========================================================
# TESTE DE CONECTIVIDADE VIA SSH TUNNEL
# =========================================================

TUNNEL_HOST = "127.0.0.1"

# PostgreSQL
try:
    import psycopg
    conn = psycopg.connect(
        host=TUNNEL_HOST,
        port=55432,
        dbname="postgres",
        user="postgres",
        password="postgres",
        connect_timeout=5
    )
    conn.close()
    print("PostgreSQL via túnel: conexão OK")
except Exception as e:
    print("PostgreSQL via túnel: falha na conexão")
    print(type(e).__name__, e)

# Cassandra
try:
    from cassandra.cluster import Cluster
    cluster = Cluster(contact_points=[TUNNEL_HOST], port=59042)
    session = cluster.connect()
    session.shutdown()
    cluster.shutdown()
    print("Cassandra via túnel: conexão OK")
except Exception as e:
    print("Cassandra via túnel: falha na conexão")
    print(type(e).__name__, e)


#mongoDB

# =========================================================
# TESTE DE CONECTIVIDADE VIA SSH TUNNEL — MONGODB
# =========================================================

from pymongo import MongoClient

try:
    client = MongoClient(
        host="127.0.0.1",
        port=57017,
        username="mongo",
        password="mongo",
        authSource="admin",
        serverSelectionTimeoutMS=5000
    )
    client.admin.command("ping")
    print("MongoDB via túnel: conexão OK")
    client.close()
except Exception as e:
    print("MongoDB via túnel: falha na conexão")
    print(type(e).__name__, e)

PostgreSQL via túnel: conexão OK
Cassandra via túnel: conexão OK
MongoDB via túnel: conexão OK


In [25]:
# =========================================================
# BLOCO A2.4 — EXECUTAR CARGA REAL NOS BANCOS
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

required_names = [
    "hubara_imdb_all_postgres_materialization",
    "all_postgres_sql_path",
    "ensure_postgres_database_exists",
    "execute_postgres_sql_script",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

# ---------------------------------------------------------
# 1) Recuperar specs físicos
# ---------------------------------------------------------
user_db_spec = hubara_imdb_all_postgres_materialization.fragment_to_db["UserFragment"]
content_db_spec = hubara_imdb_all_postgres_materialization.fragment_to_db["ContentFragment"]

print("User DB Spec:", user_db_spec)
print("Content DB Spec:", content_db_spec)

# ---------------------------------------------------------
# 2) Verificação simples de consistência
# Neste experimento, os dois fragmentos devem apontar para
# o mesmo PostgreSQL e para o mesmo database.
# ---------------------------------------------------------
if (
    user_db_spec.engine != "PostgreSQL" or
    content_db_spec.engine != "PostgreSQL" or
    user_db_spec.host != content_db_spec.host or
    user_db_spec.port != content_db_spec.port or
    user_db_spec.database_name != content_db_spec.database_name
):
    raise ValueError(
        "No experimento All PostgreSQL, UserFragment e ContentFragment "
        "devem apontar para o mesmo PostgreSQL/database."
    )

# ---------------------------------------------------------
# 3) Garantir database PostgreSQL
# ---------------------------------------------------------
ensure_postgres_database_exists(user_db_spec)

# ---------------------------------------------------------
# 4) Executar script SQL consolidado no PostgreSQL
# ---------------------------------------------------------
execute_postgres_sql_script(user_db_spec, all_postgres_sql_path)

print("\nCarga real finalizada em PostgreSQL.")

User DB Spec: PhysicalDBSpec(model='relational', engine='PostgreSQL', host='127.0.0.1', port=55432, database_name='imdb_all_postgres_db', username='postgres', password='postgres', connection_uri=None, options={})
Content DB Spec: PhysicalDBSpec(model='relational', engine='PostgreSQL', host='127.0.0.1', port=55432, database_name='imdb_all_postgres_db', username='postgres', password='postgres', connection_uri=None, options={})
Database PostgreSQL criado: imdb_all_postgres_db
Script SQL executado com sucesso em PostgreSQL: exports/imdb_hubara_all_postgres/sql/all_postgres_combined.sql

Carga real finalizada em PostgreSQL.


In [26]:
# =========================================================
# BLOCO A2.5 — VALIDAR CONTAGEM DAS TABELAS
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

# ---------------------------------------------------------
# 1) Montar lista completa de tabelas esperadas
# ---------------------------------------------------------
postgres_tables = (
    list(user_payload_all_postgres.physical_tables.keys()) +
    list(content_payload_all_postgres.physical_tables.keys())
)

# Remove duplicatas, se houver
postgres_tables = list(dict.fromkeys(postgres_tables))

# ---------------------------------------------------------
# 2) Verificar contagens no PostgreSQL
# ---------------------------------------------------------
postgres_counts_df = verify_postgres_table_counts(user_db_spec, postgres_tables)

print("Contagens no PostgreSQL (All PostgreSQL):")
display(postgres_counts_df)

Contagens no PostgreSQL (All PostgreSQL):


,table_name,row_count
0,users,100
1,persons,120
2,genres,10
3,watchitems,80
4,movies,34
5,series,15
6,episodes,31
7,roles,290
8,rates,547


In [27]:
# =========================================================
# BLOCO B1 — FUNÇÕES DE BENCHMARK E ESTATÍSTICAS MELHORADAS
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

import time
import random
import pandas as pd
import numpy as np


# ---------------------------------------------------------
# 1) Gerar parâmetros válidos para cada query
# ---------------------------------------------------------
def build_benchmark_parameter_pool(user_payload, content_payload, seed: int = 42):
    users_df = user_payload.physical_tables["users"]
    watchitems_df = content_payload.physical_tables["watchitems"]
    roles_df = content_payload.physical_tables["roles"]

    # -----------------------------
    # Q4: parâmetros vindos de watchitems
    # Precisamos de:
    # - genre_id
    # - item_type
    # - release_year
    # -----------------------------
    q4_df = watchitems_df[
        ["genre_id", "item_type", "release_year"]
    ].drop_duplicates().copy()

    q4_df["genre_id"] = q4_df["genre_id"].astype(int)
    q4_df["item_type"] = q4_df["item_type"].astype(str)
    q4_df["release_year"] = q4_df["release_year"].astype(int)

    # -----------------------------
    # Q5: parâmetros vindos de roles
    # Precisamos de:
    # - watchitem_id
    # - role_type
    # -----------------------------
    q5_df = roles_df[
        ["watchitem_id", "role_type"]
    ].drop_duplicates().copy()

    q5_df["watchitem_id"] = q5_df["watchitem_id"].astype(int)
    q5_df["role_type"] = q5_df["role_type"].astype(str)

    pool = {
        "Q1_Login": users_df["username"].astype(str).tolist(),
        "Q2_SimpleSearch": watchitems_df["title"].astype(str).tolist(),
        "Q4_Recommendation": q4_df.to_dict(orient="records"),
        "Q5_AllPersonsOfTypeForWatchItem": q5_df.to_dict(orient="records"),
    }

    return pool


# ---------------------------------------------------------
# 2) Média sem extremos
# ---------------------------------------------------------
def trimmed_mean(values, trim_each_side: int = 1) -> float:
    """
    Remove os menores e maiores tempos antes de calcular a média.

    Exemplo:
    valores = [10, 11, 12, 13, 100]
    trim_each_side = 1
    remove 10 e 100 -> média de [11, 12, 13]
    """
    arr = np.array(values, dtype=float)

    if len(arr) == 0:
        return np.nan

    arr = np.sort(arr)

    # Se não houver dados suficientes para remover extremos,
    # cai para média normal.
    if len(arr) <= 2 * trim_each_side:
        return float(np.mean(arr))

    trimmed = arr[trim_each_side: len(arr) - trim_each_side]
    return float(np.mean(trimmed))


# ---------------------------------------------------------
# 3) Funções auxiliares para percentis
# ---------------------------------------------------------
def p95(x):
    return float(np.percentile(x, 95))

def p99(x):
    return float(np.percentile(x, 99))


# ---------------------------------------------------------
# 4) Resumo do benchmark com estatísticas extras
# ---------------------------------------------------------
def summarize_benchmark_results(
    result_df: pd.DataFrame,
    trim_each_side: int = 1,
    separate_by_phase: bool = True
) -> pd.DataFrame:
    """
    Gera estatísticas agregadas por query.
    Se separate_by_phase=True, separa cold e hot.
    """

    if result_df.empty:
        return pd.DataFrame()

    ok_df = result_df[result_df["success"] == True].copy()

    group_cols = ["query_name", "fragment_name", "db_engine"]
    if separate_by_phase:
        group_cols.append("benchmark_phase")

    summary = (
        ok_df.groupby(group_cols, as_index=False)
        .agg(
            runs=("latency_ms", "count"),
            avg_ms=("latency_ms", "mean"),
            median_ms=("latency_ms", "median"),
            min_ms=("latency_ms", "min"),
            max_ms=("latency_ms", "max"),
            std_ms=("latency_ms", "std"),
            var_ms=("latency_ms", "var"),
            p95_ms=("latency_ms", p95),
            p99_ms=("latency_ms", p99),
        )
    )

    summary["std_ms"] = summary["std_ms"].fillna(0.0)
    summary["var_ms"] = summary["var_ms"].fillna(0.0)

    summary["range_ms"] = summary["max_ms"] - summary["min_ms"]

    summary["cv_pct"] = np.where(
        summary["avg_ms"] > 0,
        (summary["std_ms"] / summary["avg_ms"]) * 100.0,
        np.nan
    )

    trimmed_rows = []
    for keys, group in ok_df.groupby(group_cols):
        latencies = group["latency_ms"].tolist()
        trimmed_avg = trimmed_mean(latencies, trim_each_side=trim_each_side)

        if not isinstance(keys, tuple):
            keys = (keys,)

        row = dict(zip(group_cols, keys))
        row["avg_trimmed_ms"] = trimmed_avg
        trimmed_rows.append(row)

    trimmed_df = pd.DataFrame(trimmed_rows)

    summary = summary.merge(trimmed_df, on=group_cols, how="left")

    summary = summary.sort_values(group_cols).reset_index(drop=True)

    return summary

In [28]:
# =========================================================
# BLOCO B2 — ABRIR CONEXÕES DE BENCHMARK
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

# Reusa funções já criadas antes:
# - pg_connect(...)

def open_benchmark_connections(user_db_spec, content_db_spec):
    """
    Abre conexão persistente para o benchmark.

    No experimento All PostgreSQL, os dois fragmentos apontam para o
    mesmo PostgreSQL/database, então uma única conexão é suficiente.
    """
    # Verificação simples de consistência
    if (
        user_db_spec.engine != "PostgreSQL" or
        content_db_spec.engine != "PostgreSQL" or
        user_db_spec.host != content_db_spec.host or
        user_db_spec.port != content_db_spec.port or
        user_db_spec.database_name != content_db_spec.database_name
    ):
        raise ValueError(
            "No experimento All PostgreSQL, UserFragment e ContentFragment "
            "devem apontar para o mesmo PostgreSQL/database."
        )

    pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
    pg_conn.autocommit = True

    return pg_conn


def close_benchmark_connections(pg_conn):
    """
    Fecha a conexão PostgreSQL de forma limpa.
    """
    try:
        if pg_conn is not None:
            pg_conn.close()
    except Exception:
        pass

In [29]:
# =========================================================
# BLOCO B3 — EXECUÇÃO REAL DAS QUERIES DO WORKLOAD
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

# ---------------------------------------------------------
# Q1 — Login (PostgreSQL)
# RETURN password FROM User WHERE username = ?
# ---------------------------------------------------------
def run_q1_login_postgres(pg_conn, username: str):
    sql = """
    SELECT password
    FROM users
    WHERE username = %s
    LIMIT 1;
    """
    with pg_conn.cursor() as cur:
        cur.execute(sql, (username,))
        return cur.fetchone()


# ---------------------------------------------------------
# Q2 — Simple search (PostgreSQL)
# RETURN ALL FROM WatchItem WHERE title = ?
# Usa a tabela canônica: watchitems
# ---------------------------------------------------------
def run_q2_simple_search_postgres(pg_conn, title: str):
    sql = """
    SELECT
        watchitem_id,
        title,
        release_year,
        avg_rating,
        genre_id,
        item_type
    FROM watchitems
    WHERE title = %s;
    """
    with pg_conn.cursor() as cur:
        cur.execute(sql, (str(title),))
        return cur.fetchall()


# ---------------------------------------------------------
# Q3 — Add entities and relationships (PostgreSQL)
# Inserimos:
# - uma nova pessoa em persons
# - um novo vínculo em roles ligando a pessoa a um watchitem existente
# ---------------------------------------------------------
def run_q3_add_entities_and_relationships_postgres(
    pg_conn,
    person_id: int,
    person_name: str,
    birth_date: str,
    gender: str,
    role_id: int,
    watchitem_id: int,
    role_type: str
):
    insert_person_sql = """
    INSERT INTO persons (person_id, name, date_of_birth, gender)
    VALUES (%s, %s, %s, %s);
    """

    insert_role_sql = """
    INSERT INTO roles (role_id, person_id, watchitem_id, role_type)
    VALUES (%s, %s, %s, %s);
    """

    with pg_conn.cursor() as cur:
        cur.execute(
            insert_person_sql,
            (int(person_id), str(person_name), str(birth_date), str(gender))
        )
        cur.execute(
            insert_role_sql,
            (int(role_id), int(person_id), int(watchitem_id), str(role_type))
        )

    return True


# ---------------------------------------------------------
# Q4 — Recommendation query (PostgreSQL)
# Usa as tabelas canônicas:
# - watchitems
# - genres
#
# Filtro:
# - genre_id
# - item_type
# - release_year entre year_low e year_high
# ---------------------------------------------------------
def run_q4_recommendation_postgres(
    pg_conn,
    genre_id: int,
    item_type: str,
    year_low: int,
    year_high: int
):
    sql = """
    SELECT
        w.watchitem_id,
        w.title,
        w.release_year,
        w.avg_rating,
        w.genre_id,
        g.name AS genre_name,
        w.item_type
    FROM watchitems w
    LEFT JOIN genres g
        ON w.genre_id = g.genre_id
    WHERE w.genre_id = %s
      AND w.item_type = %s
      AND w.release_year >= %s
      AND w.release_year <= %s
    ORDER BY w.release_year, w.watchitem_id;
    """
    with pg_conn.cursor() as cur:
        cur.execute(
            sql,
            (int(genre_id), str(item_type), int(year_low), int(year_high))
        )
        return cur.fetchall()


# ---------------------------------------------------------
# Q5 — All persons of type for a watch item (PostgreSQL)
# Usa as tabelas canônicas:
# - roles
# - persons
# ---------------------------------------------------------
def run_q5_all_persons_for_watchitem_postgres(
    pg_conn,
    watchitem_id: int,
    role_type: str
):
    sql = """
    SELECT
        r.watchitem_id,
        r.role_type,
        p.person_id,
        p.name,
        p.date_of_birth,
        p.gender
    FROM roles r
    INNER JOIN persons p
        ON r.person_id = p.person_id
    WHERE r.watchitem_id = %s
      AND r.role_type = %s
    ORDER BY p.person_id;
    """
    with pg_conn.cursor() as cur:
        cur.execute(sql, (int(watchitem_id), str(role_type)))
        return cur.fetchall()

In [30]:
# =========================================================
# BLOCO B4 — EXECUTOR COMPLETO DO BENCHMARK COM COLD/HOT
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

def run_hubara_imdb_benchmark_all_postgres(
    user_db_spec,
    content_db_spec,
    user_payload,
    content_payload,
    cold_repetitions: int = 3,
    hot_repetitions: int = 20,
    seed: int = 42
) -> BenchmarkResult:

    rng = random.Random(seed)
    parameter_pool = build_benchmark_parameter_pool(
        user_payload=user_payload,
        content_payload=content_payload,
        seed=seed
    )

    max_person_id = int(content_payload.physical_tables["persons"]["person_id"].max())
    max_role_id = int(content_payload.physical_tables["roles"]["role_id"].max())
    available_watchitem_ids = content_payload.physical_tables["watchitems"]["watchitem_id"].tolist()

    benchmark_result = BenchmarkResult(
        scenario_name="IMDb",
        recommendation_name="Hubara_IMDb_AllPostgreSQL"
    )

    def append_result(
        query_name,
        fragment_name,
        db_engine,
        run_id,
        benchmark_phase,
        start_time,
        success,
        error_message
    ):
        latency_ms = (time.perf_counter() - start_time) * 1000.0

        benchmark_result.query_results.append(
            QueryExecutionResult(
                query_name=query_name,
                fragment_name=fragment_name,
                db_engine=db_engine,
                run_id=run_id,
                benchmark_phase=benchmark_phase,
                latency_ms=latency_ms,
                success=success,
                error_message=error_message,
            )
        )

    # =====================================================
    # COLD RUNS
    # Regra prática:
    # - cada execução abre e fecha conexão
    # =====================================================

    # -----------------------------------------------------
    # Q1 — PostgreSQL
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        username = str(rng.choice(parameter_pool["Q1_Login"]))

        start = time.perf_counter()
        success = True
        error_message = None

        pg_conn = None
        try:
            pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
            pg_conn.autocommit = True
            _ = run_q1_login_postgres(pg_conn, username)
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(pg_conn)

        append_result(
            "Q1_Login", "UserFragment", "PostgreSQL",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q2 — PostgreSQL
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        title = str(rng.choice(parameter_pool["Q2_SimpleSearch"]))

        start = time.perf_counter()
        success = True
        error_message = None

        pg_conn = None
        try:
            pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
            pg_conn.autocommit = True
            _ = run_q2_simple_search_postgres(pg_conn, title)
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(pg_conn)

        append_result(
            "Q2_SimpleSearch", "ContentFragment", "PostgreSQL",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q3 — PostgreSQL
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        person_id = max_person_id + run_id
        role_id = max_role_id + run_id
        watchitem_id = int(rng.choice(available_watchitem_ids))
        person_name = f"Benchmark Person Cold {person_id}"
        birth_date = "1990-01-01"
        gender = "M"
        role_type = str(rng.choice(["Actor", "Director", "Producer"]))

        start = time.perf_counter()
        success = True
        error_message = None

        pg_conn = None
        try:
            pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
            pg_conn.autocommit = True
            _ = run_q3_add_entities_and_relationships_postgres(
                pg_conn=pg_conn,
                person_id=person_id,
                person_name=person_name,
                birth_date=birth_date,
                gender=gender,
                role_id=role_id,
                watchitem_id=watchitem_id,
                role_type=role_type
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(pg_conn)

        append_result(
            "Q3_AddEntitiesAndRelationships", "ContentFragment", "PostgreSQL",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q4 — PostgreSQL
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        row = rng.choice(parameter_pool["Q4_Recommendation"])

        genre_id = int(row["genre_id"])
        item_type = str(row["item_type"])
        center_year = int(row["release_year"])

        year_low = center_year - 10
        year_high = center_year + 10

        start = time.perf_counter()
        success = True
        error_message = None

        pg_conn = None
        try:
            pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
            pg_conn.autocommit = True
            _ = run_q4_recommendation_postgres(
                pg_conn=pg_conn,
                genre_id=genre_id,
                item_type=item_type,
                year_low=year_low,
                year_high=year_high
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(pg_conn)

        append_result(
            "Q4_Recommendation", "ContentFragment", "PostgreSQL",
            run_id, "cold", start, success, error_message
        )

    # -----------------------------------------------------
    # Q5 — PostgreSQL
    # -----------------------------------------------------
    for run_id in range(1, cold_repetitions + 1):
        row = rng.choice(parameter_pool["Q5_AllPersonsOfTypeForWatchItem"])

        watchitem_id = int(row["watchitem_id"])
        role_type = str(row["role_type"])

        start = time.perf_counter()
        success = True
        error_message = None

        pg_conn = None
        try:
            pg_conn = pg_connect(user_db_spec.database_name, user_db_spec)
            pg_conn.autocommit = True
            _ = run_q5_all_persons_for_watchitem_postgres(
                pg_conn=pg_conn,
                watchitem_id=watchitem_id,
                role_type=role_type
            )
        except Exception as e:
            success = False
            error_message = str(e)
        finally:
            close_benchmark_connections(pg_conn)

        append_result(
            "Q5_AllPersonsOfTypeForWatchItem", "ContentFragment", "PostgreSQL",
            run_id, "cold", start, success, error_message
        )

    # =====================================================
    # HOT RUNS
    # Regra prática:
    # - reaproveita conexão aberta
    # =====================================================
    pg_conn = open_benchmark_connections(user_db_spec, content_db_spec)

    try:
        # -------------------------------------------------
        # Q1 — PostgreSQL
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            username = str(rng.choice(parameter_pool["Q1_Login"]))

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q1_login_postgres(pg_conn, username)
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q1_Login", "UserFragment", "PostgreSQL",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q2 — PostgreSQL
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            title = str(rng.choice(parameter_pool["Q2_SimpleSearch"]))

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q2_simple_search_postgres(pg_conn, title)
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q2_SimpleSearch", "ContentFragment", "PostgreSQL",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q3 — PostgreSQL
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            person_id = max_person_id + cold_repetitions + run_id
            role_id = max_role_id + cold_repetitions + run_id
            watchitem_id = int(rng.choice(available_watchitem_ids))
            person_name = f"Benchmark Person Hot {person_id}"
            birth_date = "1990-01-01"
            gender = "M"
            role_type = str(rng.choice(["Actor", "Director", "Producer"]))

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q3_add_entities_and_relationships_postgres(
                    pg_conn=pg_conn,
                    person_id=person_id,
                    person_name=person_name,
                    birth_date=birth_date,
                    gender=gender,
                    role_id=role_id,
                    watchitem_id=watchitem_id,
                    role_type=role_type
                )
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q3_AddEntitiesAndRelationships", "ContentFragment", "PostgreSQL",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q4 — PostgreSQL
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            row = rng.choice(parameter_pool["Q4_Recommendation"])

            genre_id = int(row["genre_id"])
            item_type = str(row["item_type"])
            center_year = int(row["release_year"])

            year_low = center_year - 10
            year_high = center_year + 10

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q4_recommendation_postgres(
                    pg_conn=pg_conn,
                    genre_id=genre_id,
                    item_type=item_type,
                    year_low=year_low,
                    year_high=year_high
                )
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q4_Recommendation", "ContentFragment", "PostgreSQL",
                run_id, "hot", start, success, error_message
            )

        # -------------------------------------------------
        # Q5 — PostgreSQL
        # -------------------------------------------------
        for run_id in range(1, hot_repetitions + 1):
            row = rng.choice(parameter_pool["Q5_AllPersonsOfTypeForWatchItem"])

            watchitem_id = int(row["watchitem_id"])
            role_type = str(row["role_type"])

            start = time.perf_counter()
            success = True
            error_message = None

            try:
                _ = run_q5_all_persons_for_watchitem_postgres(
                    pg_conn=pg_conn,
                    watchitem_id=watchitem_id,
                    role_type=role_type
                )
            except Exception as e:
                success = False
                error_message = str(e)

            append_result(
                "Q5_AllPersonsOfTypeForWatchItem", "ContentFragment", "PostgreSQL",
                run_id, "hot", start, success, error_message
            )

    finally:
        close_benchmark_connections(pg_conn)

    return benchmark_result

In [31]:
# =========================================================
# BLOCO B5 — RODAR BENCHMARK, RESUMIR E EXPORTAR CSV
# ALTERNATIVA: ALL POSTGRESQL
# =========================================================

# ---------------------------------------------------------
# 0) Verificando se está tudo OK
# ---------------------------------------------------------
required_names = [
    "user_db_spec",
    "content_db_spec",
    "user_payload_all_postgres",
    "content_payload_all_postgres",
    "run_hubara_imdb_benchmark_all_postgres",
    "summarize_benchmark_results",
]

for name in required_names:
    if name not in globals():
        raise NameError(
            f"'{name}' não está definido. Rode primeiro os blocos anteriores."
        )

from pathlib import Path

# ---------------------------------------------------------
# 1) Parâmetros do benchmark
# ---------------------------------------------------------
COLD_REPETITIONS = 3
HOT_REPETITIONS = 20
TRIM_EACH_SIDE = 1

# ---------------------------------------------------------
# 2) Executar benchmark
# ---------------------------------------------------------
benchmark_result = run_hubara_imdb_benchmark_all_postgres(
    user_db_spec=user_db_spec,
    content_db_spec=content_db_spec,
    user_payload=user_payload_all_postgres,
    content_payload=content_payload_all_postgres,
    cold_repetitions=COLD_REPETITIONS,
    hot_repetitions=HOT_REPETITIONS,
    seed=42
)

# ---------------------------------------------------------
# 3) Converter resultados para DataFrame
# ---------------------------------------------------------
benchmark_df = benchmark_result.to_dataframe()

summary_df = summarize_benchmark_results(
    result_df=benchmark_df,
    trim_each_side=TRIM_EACH_SIDE,
    separate_by_phase=True
)

failures_df = benchmark_df[benchmark_df["success"] == False].copy()

# ---------------------------------------------------------
# 4) Mostrar resultados
# ---------------------------------------------------------
print("Resultados brutos do benchmark:")
display(benchmark_df.head(30))

print("Resumo estatístico do benchmark:")
display(summary_df)

print("Falhas, se existirem:")
display(failures_df)

# ---------------------------------------------------------
# 5) Exportar resultados para CSV
# ---------------------------------------------------------
results_dir = Path("exports/benchmark_results_all_postgres")
results_dir.mkdir(parents=True, exist_ok=True)

raw_csv_path = results_dir / "hubara_imdb_all_postgres_benchmark_raw.csv"
summary_csv_path = results_dir / "hubara_imdb_all_postgres_benchmark_summary.csv"
failures_csv_path = results_dir / "hubara_imdb_all_postgres_benchmark_failures.csv"

benchmark_df.to_csv(raw_csv_path, index=False)
summary_df.to_csv(summary_csv_path, index=False)
failures_df.to_csv(failures_csv_path, index=False)

print("\nArquivos CSV gerados:")
print(raw_csv_path)
print(summary_csv_path)
print(failures_csv_path)

Resultados brutos do benchmark:


,scenario_name,recommendation_name,query_name,fragment_name,db_engine,run_id,benchmark_phase,latency_ms,success,error_message
0,IMDb,Hubara_IMDb_AllPostgreSQL,Q1_Login,UserFragment,PostgreSQL,1,cold,22.694382,True,None
1,IMDb,Hubara_IMDb_AllPostgreSQL,Q1_Login,UserFragment,PostgreSQL,2,cold,17.866768,True,None
2,IMDb,Hubara_IMDb_AllPostgreSQL,Q1_Login,UserFragment,PostgreSQL,3,cold,18.062903,True,None
3,IMDb,Hubara_IMDb_AllPostgreSQL,Q2_SimpleSearch,ContentFragment,PostgreSQL,1,cold,16.463871,True,None
4,IMDb,Hubara_IMDb_AllPostgreSQL,Q2_SimpleSearch,ContentFragment,PostgreSQL,2,cold,15.645865,True,None
5,IMDb,Hubara_IMDb_AllPostgreSQL,Q2_SimpleSearch,ContentFragment,PostgreSQL,3,cold,17.532582,True,None
6,IMDb,Hubara_IMDb_AllPostgreSQL,Q3_AddEntitiesAndRelationships,ContentFragment,PostgreSQL,1,cold,22.990599,True,None
7,IMDb,Hubara_IMDb_AllPostgreSQL,Q3_AddEntitiesAndRelationships,ContentFragment,PostgreSQL,2,cold,24.274813,True,None
8,IMDb,Hubara_IMDb_AllPostgreSQL,Q3_AddEntitiesAndRelationships,ContentFragment,PostgreSQL,3,cold,26.253222,True,None
9,IMDb,Hubara_IMDb_AllPostgreSQL,Q4_Recommendation,ContentFragment,PostgreSQL,1,cold,20.632392,True,None


Resumo estatístico do benchmark:


,query_name,fragment_name,db_engine,benchmark_phase,runs,avg_ms,median_ms,min_ms,max_ms,std_ms,var_ms,p95_ms,p99_ms,range_ms,cv_pct,avg_trimmed_ms
0,Q1_Login,UserFragment,PostgreSQL,cold,3,19.541351,18.062903,17.866768,22.694382,2.732365,7.465820,22.231234,22.601752,4.827614,13.982479,18.062903
1,Q1_Login,UserFragment,PostgreSQL,hot,20,1.580685,1.205103,0.780765,4.732932,1.016872,1.034029,3.382860,4.462918,3.952167,64.331131,1.450000
2,Q2_SimpleSearch,ContentFragment,PostgreSQL,cold,3,16.547439,16.463871,15.645865,17.532582,0.946131,0.895163,17.425711,17.511208,1.886717,5.717686,16.463871
3,Q2_SimpleSearch,ContentFragment,PostgreSQL,hot,20,1.100639,1.008174,0.743694,2.011609,0.290707,0.084511,1.557367,1.920761,1.267915,26.412577,1.069860
4,Q3_AddEntitiesAndRelationships,ContentFragment,PostgreSQL,cold,3,24.506211,24.274813,22.990599,26.253222,1.643574,2.701336,26.055381,26.213654,3.262623,6.706765,24.274813
5,Q3_AddEntitiesAndRelationships,ContentFragment,PostgreSQL,hot,20,5.181804,3.921252,3.041539,9.344572,2.154870,4.643466,8.362662,9.148190,6.303033,41.585331,5.069443
6,Q4_Recommendation,ContentFragment,PostgreSQL,cold,3,17.160338,16.366419,14.482202,20.632392,3.151022,9.928938,20.205795,20.547072,6.150190,18.362236,16.366419
7,Q4_Recommendation,ContentFragment,PostgreSQL,hot,20,2.309299,1.604516,1.015210,6.719021,1.567860,2.458184,4.781003,6.331417,5.703811,67.893306,2.136209
8,Q5_AllPersonsOfTypeForWatchItem,ContentFragment,PostgreSQL,cold,3,20.927435,20.060891,18.060252,24.661161,3.384697,11.456173,24.201134,24.569156,6.600909,16.173492,20.060891
9,Q5_AllPersonsOfTypeForWatchItem,ContentFragment,PostgreSQL,hot,20,1.392457,1.326476,0.842176,2.164822,0.387061,0.149816,1.963466,2.124551,1.322646,27.796950,1.380119


Falhas, se existirem:


,scenario_name,recommendation_name,query_name,fragment_name,db_engine,run_id,benchmark_phase,latency_ms,success,error_message



Arquivos CSV gerados:
exports/benchmark_results_all_postgres/hubara_imdb_all_postgres_benchmark_raw.csv
exports/benchmark_results_all_postgres/hubara_imdb_all_postgres_benchmark_summary.csv
exports/benchmark_results_all_postgres/hubara_imdb_all_postgres_benchmark_failures.csv
